# 1 Preprocessing Single File

Sanity-check the ten spike encoders on a single SEGY file.

Paths are imported from `config.py` at the repository root.
Run the notebooks in numbered order; see the repository README.

In [ ]:
"""
DAS Microseismic SNN Encoding — Single File Preprocessing & Visualization
==========================================================================
Final encoder set: 3 baselines + 7 proposed (novelty) encoders
All parameters updated from confirmed sweep best results (400 event +
400 noise files, FRISCO FORGE 2024 dataset).

  BASELINES (established SNN encoding literature):
    [B1] Rate:      thr=0.85
         Inspired by: Hubel & Wiesel (1959); Mahowald & Douglas (1991)
    [B2] Phase:     thr=0.85
         Inspired by: Thorpe et al. (2001); Bohte et al. (2002)
    [B3] TTFS:      thr=0.85
         Inspired by: Thorpe, Fize & Marlot (1996); Guo et al.
         Frontiers Neurosci (2021)
         NOTE: Converges to Phase on DAS sharp-onset data (sweep finding)

  PROPOSED (DAS-specific novel contributions):
    [P1] Delta-Mod: thr=0.85, step_size=0.18
         Adapted from: Inose et al. (1962); Petro et al. TNNLS (2020)
         DAS modification: per-trace step_size scaling for amplitude decay
    [P2] PDE:       kappa_phi=0.5, amp_kappa=3.0
         Adapted from: Gabor (1946) analytic signal; Bello et al.
         IEEE Trans. Speech Audio (2005) phase onset detection
         DAS modification: MAD-adaptive dual threshold
    [P3] ATDE:      alpha=3.0, tau_ms=165.0
         Adapted from: Jayant (1970) adaptive delta modulation;
         online EMA noise estimation (adaptive filter theory)
         DAS modification: EMA τ=165ms tracks DAS non-stationary noise
    [P4] MASTE:     lags=[1,3,8], alpha=2.5
         Adapted from: multi-resolution signal analysis; coincidence
         detection (König, Engel & Singer 1996)
         DAS modification: ATDE per lag + majority vote for DAS wavefronts
    [P5] ST-MW:     thr_t=0.18, thr_s=0.45, wt_ms=1.0, ws=8
         Adapted from: Moving-Window (MW) encoding, Petro et al.
         TNNLS (2020) → extended from 1D to 2D spatio-temporal for DAS
         DAS modification: joint temporal × spatial amplitude-change gate
    [P6] AMSTE:     alpha=3.0, lags=[1,3,8], min_votes=2, ws=16, thr_s=0.5
         Novel DAS-ASE encoder. Combines 4 DAS-specific properties:
         (1) per-channel MAD-adaptive threshold (PhaseNet-DAS inspired)
         (2) bidirectional polarity detection (Frontiers Neurosci 2021)
         (3) multi-scale temporal voting (arXiv 2407.09260, 2024)
         (4) spatial neighbourhood coherence gate (extends ST-MW)
    [P7] SFBE:      ratio_thr=10.0, sta_ms=8.0
         Adapted from: FEEL-SNN NeurIPS (2024) frequency encoding;
         STA/LTA onset detection, Allen (1978) BSSA
         DAS modification: seismic-physics sub-bands + fixed pre-event
         baseline LTA replaces cumulative LTA (prevents contamination)
         Evaluation: sparsity discrimination ratio (sp_ev/sp_no=6.4×)
         rather than windowed SNR (time-multiplexed output)
"""

import segyio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal  import butter, filtfilt, hilbert
from scipy.ndimage import maximum_filter1d, minimum_filter1d
from scipy.stats   import pearsonr
import os

# =============================================================================
# CONFIGURATION — all params from confirmed sweep (400ev + 400no files)
# =============================================================================
FILE_PATH  = config.SINGLE_FILE_EVENT
OUTPUT_DIR = config.SINGLE_OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

RNG_SEED = 42

# ── Preprocessing ─────────────────────────────────────────────────────────────
FORCE_DT_MS        = 0.5   # confirmed: FRISCO FORGE 2024, 2kHz DAS interrogator
FILTER_LOW_HZ      = 50
FILTER_HIGH_HZ     = 250
EXTRACT_WINDOW     = True
WINDOW_START_MS    = 0
WINDOW_END_MS      = 1000
TRACE_SPACING_M    = 4.0
ENERGY_PER_SPIKE_PJ = 23.6
EVENT_START_MS     = 380.0
EVENT_END_MS       = 560.0
# NOTE: Single-file diagnostic window only.
# Batch experiments use per-file labels from CSV — not this hard-coded window.
PRE_EVENT_END_MS   = 200.0    # SFBE fixed LTA baseline window (before event)
DISPLAY_TRACE      = 50

# ── Shared (Rate, Phase, TTFS) ────────────────────────────────────────────────
THRESHOLD         = 0.85       # sweep confirmed: 0.85
TIME_WINDOW_MS    = 1.0

# ── Delta-Mod ─────────────────────────────────────────────────────────────────
DELTA_MOD_THRESHOLD = 0.85     # sweep: 0.85
DELTA_MOD_STEP_SIZE = 0.18     # sweep: 0.18  ← was 0.08, updated

# ── PDE ───────────────────────────────────────────────────────────────────────
PDE_KAPPA           = 0.5      # sweep: 0.5   ← was 1.0, updated
PDE_AMP_KAPPA       = 3.0      # sweep: 3.0   ← was 2.0, updated
PDE_MIN_DELTA_PHI   = 0.10
PDE_AMP_SMOOTH_MS   = 5.0

# ── ATDE ──────────────────────────────────────────────────────────────────────
ATDE_ALPHA          = 3.0      # sweep: 3.0   ← was 2.5, updated
ATDE_BETA           = 0.01
ATDE_NOISE_WINDOW   = 50
ATDE_EMA_TAU_MS     = 165.0    # sweep: 165.0 ← was 35.0, updated

# ── MASTE ─────────────────────────────────────────────────────────────────────
MASTE_LAGS          = [1, 3, 8]  # sweep: [1,3,8] ✓
MASTE_ALPHA         = 2.5        # sweep: 2.5   ✓
MASTE_TAU_MS        = 50.0

# ── ST-MW ─────────────────────────────────────────────────────────────────────
# Adapted from: Moving-Window (MW), Petro et al. TNNLS 2020
# DAS extension: 1D temporal → 2D joint spatio-temporal condition
STMW_THR_TEMPORAL   = 0.18     # sweep: 0.18  ← was 0.35, updated
STMW_THR_SPATIAL    = 0.45     # sweep: 0.45  ← was 0.20, updated
STMW_WINDOW_T       = 1.0      # sweep: 1.0ms ← was 2.0ms, updated
STMW_WINDOW_S       = 8        # sweep: 8ch   ✓

# ── AMSTE (replaces ICDE — was 0/175 in target) ───────────────────────────────
# Novel DAS-ASE: Adaptive Multi-Scale Spatio-Temporal Delta Encoder
# Combines 4 DAS-specific properties (see module docstring)
AMSTE_ALPHA         = 3.0      # MAD multiplier for per-channel threshold
AMSTE_LAGS          = [1, 3, 8]  # temporal scales: 0.5, 1.5, 4.0 ms
AMSTE_MIN_VOTES     = 2        # majority vote (≥2/3 lags must agree)
AMSTE_WS            = 16       # spatial window: 16 channels (64m aperture)
AMSTE_THR_S         = 0.5      # spatial amplitude gate threshold

# ── SFBE (fixed LTA — was SNR=1.26×, now STA/LTA fixed baseline) ─────────────
# Adapted from: FEEL-SNN NeurIPS 2024 (time-multiplex retained)
# STA/LTA: Allen (1978); fixed baseline: pre-event 0–200ms window
SFBE_BANDS          = [(50, 100), (100, 150), (150, 200), (200, 250)]
SFBE_RATIO_THR      = 10.0    # sweep: 10.0 — STA/LTA detection ratio
SFBE_STA_MS         = 8.0     # sweep: 8.0ms — short-term averaging window

# ── Encoder colour map ────────────────────────────────────────────────────────
ENCODER_COLORS = {
    'Rate':      '#2196F3',   # blue
    'Phase':     '#FF9800',   # orange
    'TTFS':      '#4CAF50',   # green
    'Delta-Mod': '#9C27B0',   # purple
    'PDE':       '#F44336',   # red
    'ATDE':      '#009688',   # teal
    'MASTE':     '#E91E63',   # pink
    'ST-MW':     '#8BC34A',   # light green
    'AMSTE':     '#FF5722',   # deep orange  ← replaces ICDE
    'SFBE':      '#3F51B5',   # indigo
}


# =============================================================================
# 1. DATA LOADING
# =============================================================================
def load_segy_file(filename, force_dt_ms=FORCE_DT_MS):
    try:
        with segyio.open(filename, mode='r', ignore_geometry=True) as f:
            data = np.array([f.trace[i] for i in range(len(f.trace))])
            try:
                dt_ms = f.bin[segyio.BinField.Interval] / 1000.0
                if not (0.001 <= dt_ms <= 10.0):
                    print(f"  Header DT invalid → forced: {force_dt_ms} ms")
                    dt_ms = force_dt_ms
                else:
                    print(f" Header DT: {dt_ms} ms")
            except Exception:
                print(f"  Cannot read DT → forced: {force_dt_ms} ms")
                dt_ms = force_dt_ms
            print(f" {data.shape[0]} traces × {data.shape[1]} samples "
                  f"| {dt_ms} ms ({1000/dt_ms:.0f} Hz)")
            return data, dt_ms
    except Exception as e:
        print(f" {e}"); return None, None


# =============================================================================
# 2. PREPROCESSING (identical to batch pipeline)
# =============================================================================
def bandpass_filter(data, dt_ms, lo=FILTER_LOW_HZ, hi=FILTER_HIGH_HZ):
    fs  = 1000.0 / dt_ms
    nyq = 0.5 * fs
    b, a = butter(4, [np.clip(lo/nyq, 1e-6, 0.99),
                      np.clip(hi/nyq, 1e-6, 0.99)], btype='band')
    if data.ndim == 1:
        return filtfilt(b, a, data)
    out = np.zeros_like(data)
    for i in range(data.shape[0]):
        out[i] = filtfilt(b, a, data[i])
    return out

def extract_time_window(data, dt_ms, s_ms, e_ms):
    s = max(0, int(s_ms / dt_ms))
    e = min(data.shape[1], int(e_ms / dt_ms))
    return (data, 0, data.shape[1]) if s >= e else (data[:, s:e], s, e)

def normalize_data(data, mode='unit'):
    out = np.zeros_like(data)
    for i in range(data.shape[0]):
        t = data[i] - np.mean(data[i])
        if mode == 'zscore':
            s = np.std(t)
            out[i] = t / s if s > 0 else t
        else:
            m = np.max(np.abs(t))
            out[i] = (t / m + 1) / 2 if m > 0 else 0.5
    return out

def normalize_signed(data):
    """
    Signed normalisation → output range [-1, +1] per trace.
    Used by polarity-sensitive encoders: Delta-Mod, PDE, ATDE, MASTE,
    ST-MW, AMSTE. Preserves the sign of seismic strain (compression vs
    rarefaction), which is physically meaningful in DAS data and required
    for AMSTE's bidirectional polarity voting to function correctly.

    The [0,1] unit normalisation maps a −0.8 excursion to 0.1 (near zero),
    making it invisible to threshold-based encoders. Signed normalisation
    maps −0.8 → −0.8 so negative strain events are detected symmetrically.
    """
    out = np.zeros_like(data, dtype=np.float64)
    for i in range(data.shape[0]):
        tr = data[i] - np.mean(data[i])
        m  = np.max(np.abs(tr))
        out[i] = tr / m if m > 0 else tr
    return out


def extract_envelope(data):
    env = np.zeros_like(data)
    for i in range(data.shape[0]):
        e = np.abs(hilbert(data[i]))
        m = np.max(e)
        env[i] = e / m if m > 0 else e
    return env


# =============================================================================
# 3. ENCODERS
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# BASELINES
# ─────────────────────────────────────────────────────────────────────────────

def rate_encoding(data, dt_ms, thr=THRESHOLD, seed=RNG_SEED):
    """
    Rate Encoding — Baseline B1.
    Source/Inspiration: Hubel & Wiesel (1959); Mahowald & Douglas (1991).
    Amplitude within each 1ms window drives a Poisson spike probability.
    Windows below threshold are silent; windows above fire proportionally.
    Sweep best: thr=0.85 → SNR=3.09×, sparsity=0.46%
    """
    rng = np.random.default_rng(seed)
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr = data[t]
        for i in range(0, len(tr) - spw, spw):
            amp = np.mean(np.abs(tr[i:i+spw]))
            if amp > thr:
                out[t, i:i+spw] = (rng.random(spw) < np.clip(amp, 0, 1)).astype(float)
    sp = np.sum(out) / out.size
    print(f"   Rate      sparsity: {sp:.2%}")
    return out

def phase_encoding(data, dt_ms, thr=THRESHOLD):
    """
    Phase Encoding — Baseline B2.
    Source/Inspiration: Thorpe et al. (2001); Bohte et al. (2002).
    Amplitude maps to spike delay within each 1ms window:
        delay = (1 - amp) × (window_size - 1)
    High amplitude → early spike. At most one spike per window.
    Encodes stimulus INTENSITY as temporal position.
    Sweep best: thr=0.85 → SNR=2.71×, sparsity=0.51%
    """
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr = data[t]
        for i in range(0, len(tr) - spw, spw):
            amp = np.max(np.abs(tr[i:i+spw]))
            if amp > thr:
                delay = int(np.clip(1 - amp, 0, 1) * (spw - 1))
                idx = i + delay
                if idx < out.shape[1]:
                    out[t, idx] = 1.0
    sp = np.sum(out) / out.size
    print(f"   Phase     sparsity: {sp:.2%}")
    return out

def ttfs_encoding(data, dt_ms, thr=THRESHOLD):
    """
    Time-To-First-Spike (TTFS) Encoding — Baseline B3.
    Source/Inspiration: Thorpe, Fize & Marlot (1996) Nature;
    Guo et al. Frontiers Neuroscience (2021) comparative SNN coding study.
    NOTE: Petro et al. TNNLS (2020) is cited for temporal contrast
    encoders (Delta-Mod, ST-MW), NOT as the primary TTFS source.

    Fires at the FIRST sample within each 1ms window that crosses
    the threshold — encodes stimulus ONSET LATENCY (not intensity).
        First crossing → early in window = early P-wave arrival
        Late crossing  → slow S-wave or coda

    DAS FINDING FROM SWEEP: TTFS converges to Phase encoding on FORGE
    microseismic data. Sharp-onset P-wave arrivals mean the first
    threshold-crossing sample coincides with the peak amplitude sample
    within each analysis window. Report as paper finding, not a bug.
    Sweep best: thr=0.85 → SNR=2.71× (equal to Phase)
    """
    spw = max(1, int(TIME_WINDOW_MS / dt_ms))
    out = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr    = data[t]
        abs_tr = np.abs(tr)
        for i in range(0, len(tr) - spw, spw):
            win = abs_tr[i:i+spw]
            if not (win > thr).any():
                continue
            # FIRST sample crossing threshold (not peak amplitude)
            first = int(np.argmax(win > thr))
            idx   = i + first
            if idx < out.shape[1]:
                out[t, idx] = 1.0
    sp = np.sum(out) / out.size
    print(f"   TTFS      sparsity: {sp:.2%}  "
          f"[NOTE: converges to Phase on DAS sharp-onset data]")
    return out


# ─────────────────────────────────────────────────────────────────────────────
# PROPOSED ENCODERS
# ─────────────────────────────────────────────────────────────────────────────

def delta_modulation_encoding(data, dt_ms,
                               thr=DELTA_MOD_THRESHOLD,
                               step_size=DELTA_MOD_STEP_SIZE):
    """
    Delta Modulation Encoding — Proposed P1.
    Adapted from: Inose, Yasuda & Murakami (1962) delta modulation;
    Petro et al. TNNLS (2020) temporal contrast SNN encoders.

    DAS modification: step_size = 0.18 × tr_range (per-trace scaling).
    Original uses a fixed step. DAS channels have different amplitudes
    due to fiber attenuation — scaling to tr_range normalises for this.
    Fires when signal departs from running baseline by ≥ step_size.
    Sweep best: thr=0.85, step_size=0.18 → SNR=2.63×, sparsity=0.30%
    """
    min_interval = max(1, int(2.0 / dt_ms))
    out          = np.zeros_like(data)
    for t in range(data.shape[0]):
        tr         = data[t]
        tr_range   = np.max(tr) - np.min(tr)
        delta_step = max(step_size * tr_range, 1e-6)
        reference  = tr[0]
        last_spike = -min_interval
        for i in range(1, len(tr)):
            diff = tr[i] - reference
            if abs(diff) >= delta_step:
                n_steps    = int(abs(diff) / delta_step)
                reference += np.sign(diff) * n_steps * delta_step
                if diff > 0 and tr[i] > thr:
                    if (i - last_spike) >= min_interval:
                        out[t, i]  = 1.0
                        last_spike = i
    sp = np.sum(out) / out.size
    print(f"   Delta-Mod sparsity: {sp:.2%}")
    return out

def phase_delta_encoding(data, dt_ms,
                          kappa=PDE_KAPPA,
                          amp_kappa=PDE_AMP_KAPPA,
                          min_delta_phi=PDE_MIN_DELTA_PHI,
                          amp_smooth_ms=PDE_AMP_SMOOTH_MS):
    """
    Phase-Delta Encoding (PDE) — Proposed P2.
    Adapted from: Gabor (1946) Hilbert analytic signal; Bello et al.
    IEEE Trans. Speech Audio Processing (2005) phase-based onset detection.

    DAS modification: MAD-adaptive dual threshold replaces fixed threshold.
    Fires when BOTH instantaneous amplitude AND instantaneous phase change
    exceed their respective adaptive thresholds simultaneously.
    Joint condition rejects isolated noise (amplitude-only or phase-only).
    Sweep best: kappa_phi=0.5, amp_kappa=3.0 → SNR=2.17×, prec=0.268
    (highest precision of all 10 encoders)
    """
    n_traces, n_samples = data.shape
    out      = np.zeros_like(data)
    smooth_k = max(3, int(amp_smooth_ms / dt_ms))
    kernel   = np.ones(smooth_k) / smooth_k
    for t in range(n_traces):
        tr = data[t]
        N  = len(tr)
        X  = np.fft.fft(tr)
        Z  = np.zeros(N, dtype=complex)
        Z[0] = X[0]
        if N % 2 == 0:
            Z[1:N//2]  = 2.0 * X[1:N//2]
            Z[N//2]    = X[N//2]
        else:
            Z[1:(N+1)//2] = 2.0 * X[1:(N+1)//2]
        z       = np.fft.ifft(Z)
        env_raw = np.abs(z)
        env     = np.convolve(env_raw, kernel, mode='same')
        p25     = np.percentile(env, 25)
        mad_env = np.median(np.abs(env - np.median(env)))
        mu_amp  = p25 + amp_kappa * 1.4826 * mad_env
        conj_prod     = z[1:] * np.conj(z[:-1])
        delta_phi_arr = np.angle(conj_prod)
        abs_dphi      = np.abs(delta_phi_arr)
        median_dphi   = np.median(abs_dphi)
        mad_dphi      = np.median(np.abs(abs_dphi - median_dphi))
        delta_phi_thr = max(median_dphi + kappa * 1.4826 * mad_dphi,
                            min_delta_phi)
        for n in range(len(delta_phi_arr)):
            if env[n+1] < mu_amp:
                continue
            if abs(delta_phi_arr[n]) > delta_phi_thr:
                out[t, n+1] = 1.0
    sp = np.sum(out) / out.size
    print(f"   PDE       sparsity: {sp:.2%}  (κ_φ={kappa}, κ_a={amp_kappa})")
    return out

def adaptive_threshold_delta_encoding(data, dt_ms,
                                       alpha=ATDE_ALPHA,
                                       beta=ATDE_BETA,
                                       W=ATDE_NOISE_WINDOW,
                                       tau_ms=ATDE_EMA_TAU_MS):
    """
    Adaptive Threshold Delta Encoding (ATDE) — Proposed P3.
    Adapted from: Jayant (1970) adaptive delta modulation (Bell Syst. Tech. J.);
    exponential moving average noise estimation (adaptive filter theory;
    Haykin, 'Adaptive Filter Theory', 2002).

    DAS modification: EMA time constant τ=165ms tracks DAS non-stationary
    noise floor. Long τ prevents a seismic arrival from immediately raising
    the threshold (which would suppress the event's own spikes).
    Threshold: θ(t) = α × √σ²_ema(t) + β
    Sweep best: alpha=3.0, tau_ms=165.0 → SNR=2.22×, sparsity=0.35%
    """
    n_traces, n_samples = data.shape
    out       = np.zeros_like(data)
    alpha_ema = 1.0 - np.exp(-(dt_ms / tau_ms)) if tau_ms > 0 else 0.1
    for t in range(n_traces):
        tr    = data[t]
        diffs = np.diff(tr)
        init_len     = min(W, len(diffs))
        sigma_sq_ema = np.var(diffs[:init_len]) if init_len > 1 else 1e-6
        for n in range(1, n_samples):
            delta_x = diffs[n-1]
            if n < W:
                sigma_sq_ema = alpha_ema * delta_x**2 + (1-alpha_ema) * sigma_sq_ema
                continue
            sigma_sq_ema = alpha_ema * delta_x**2 + (1-alpha_ema) * sigma_sq_ema
            sigma_hat    = np.sqrt(sigma_sq_ema)
            theta        = alpha * max(sigma_hat, 1e-9) + beta
            if abs(delta_x) > theta:
                out[t, n] = 1.0
    sp = np.sum(out) / out.size
    print(f"   ATDE      sparsity: {sp:.2%}  (α={alpha}, τ={tau_ms}ms)")
    return out

def maste_encoding(data, dt_ms,
                   lags=None, alpha=None, tau_ms=None, min_votes=2):
    """
    Multi-lag Adaptive Spike Timing Encoding (MASTE) — Proposed P4.
    Adapted from: multi-resolution signal analysis (Mallat 1989);
    coincidence detection in neural circuits (König, Engel & Singer 1996).

    DAS modification: applies ATDE at lags [1,3,8] = [0.5, 1.5, 4.0 ms],
    retaining spikes only when ≥ min_votes=2 lags agree (majority vote).
    A real DAS seismic wavefront is coherent across multiple timescales.
    Random noise spikes are time-localised (lag-1 only) and fail the vote.
    Sweep best: lags=[1,3,8], alpha=2.5 → SNR=2.52×, prec=0.264
    """
    if lags   is None: lags   = MASTE_LAGS
    if alpha  is None: alpha  = MASTE_ALPHA
    if tau_ms is None: tau_ms = MASTE_TAU_MS
    n_traces, n_samples = data.shape
    W          = ATDE_NOISE_WINDOW
    alpha_ema  = 1.0 - np.exp(-(dt_ms / tau_ms)) if tau_ms > 0 else 0.1
    vote_count = np.zeros((n_traces, n_samples), dtype=np.int32)
    for lag in lags:
        out = np.zeros_like(data)
        for t in range(n_traces):
            tr    = data[t]
            diffs = tr[lag:] - tr[:-lag]
            init_len     = min(W, len(diffs))
            sigma_sq_ema = np.var(diffs[:init_len]) if init_len > 1 else 1e-6
            for n in range(lag, n_samples):
                delta_x = diffs[n-lag]
                if n < W + lag:
                    sigma_sq_ema = alpha_ema * delta_x**2 + (1-alpha_ema) * sigma_sq_ema
                    continue
                sigma_sq_ema = alpha_ema * delta_x**2 + (1-alpha_ema) * sigma_sq_ema
                sigma_hat    = np.sqrt(sigma_sq_ema)
                theta        = alpha * max(sigma_hat, 1e-9) + ATDE_BETA
                if abs(delta_x) > theta:
                    out[t, n] = 1.0
        sp_lag = np.sum(out) / out.size
        print(f"   MASTE [lag={lag:>2d}]  Δt={lag*dt_ms:.1f}ms  sparsity={sp_lag:.2%}")
        vote_count += out.astype(np.int32)
    fused = (vote_count >= min_votes).astype(data.dtype)
    sp = np.sum(fused) / fused.size
    print(f"   MASTE fused (≥{min_votes}/{len(lags)} votes)  sparsity={sp:.2%}")
    return fused

def stmw_encoding(data, dt_ms,
                   thr_t=STMW_THR_TEMPORAL,
                   thr_s=STMW_THR_SPATIAL,
                   wt=STMW_WINDOW_T,
                   ws=STMW_WINDOW_S):
    """
    Spatio-Temporal Moving-Window Encoding (ST-MW) — Proposed P5.
    Adapted from: Moving-Window (MW) encoding, Petro et al. TNNLS (2020).
    DAS extension: 1D temporal MW → 2D joint spatio-temporal condition.

    Original MW (1D): spike if amplitude change in time window > θ.
    ST-MW (2D): spike at (channel i, time t) only when BOTH:
      (a) Temporal: max(|x[i,t:t+Wt]|) − min(|x[i,t:t+Wt]|) > θ_t
      (b) Spatial:  max(|x[i-Ws/2:i+Ws/2, t_mid]|)
                  − min(|x[i-Ws/2:i+Ws/2, t_mid]|) > θ_s
    Spatial condition suppresses single-channel fading/erratic noise
    while preserving coherent seismic wavefronts across multiple channels.
    Sweep best: thr_t=0.18, thr_s=0.45, wt_ms=1.0, ws=8
    → SNR=5.40× (highest of all 10 encoders), prec=0.283
    """
    n_traces, n_samples = data.shape
    spw_t = max(1, int(wt / dt_ms))
    spw_s = max(2, ws)
    out   = np.zeros_like(data)
    for i in range(n_traces):
        for t in range(0, n_samples - spw_t, spw_t):
            win_t = np.abs(data[i, t:t + spw_t])
            da_t  = np.max(win_t) - np.min(win_t)
            if da_t <= thr_t:
                continue
            t_mid    = t + spw_t // 2
            ch_start = max(0, i - spw_s // 2)
            ch_end   = min(n_traces, i + spw_s // 2 + 1)
            win_s    = np.abs(data[ch_start:ch_end, t_mid])
            da_s     = np.max(win_s) - np.min(win_s)
            if da_s <= thr_s:
                continue
            peak_off = int(np.argmax(np.abs(data[i, t:t + spw_t])))
            idx      = t + peak_off
            if idx < n_samples:
                out[i, idx] = 1.0
    sp = np.sum(out) / out.size
    print(f"   ST-MW     sparsity: {sp:.2%}  "
          f"(θ_t={thr_t:.2f}, θ_s={thr_s:.2f}, Wt={wt}ms, Ws={ws}ch)")
    return out

def amste_encoding(data, dt_ms,
                    alpha=AMSTE_ALPHA,
                    lags=None,
                    min_votes=AMSTE_MIN_VOTES,
                    ws=AMSTE_WS,
                    thr_s=AMSTE_THR_S):
    """
    AMSTE: Adaptive Multi-Scale Spatio-Temporal Delta Encoder — Proposed P6.
    Novel DAS-ASE encoder. Not a direct adaptation of any single paper.
    Combines 4 DAS-specific properties absent from all other encoders:

    STEP 1 — Per-channel MAD-adaptive threshold
    ────────────────────────────────────────────
    Inspired by: PhaseNet-DAS (Nature Comms 2023) channel-specific
    noise normalisation for DAS with unknown ground coupling.
        MAD_c    = median(|X[c,:] − median(X[c,:])|)
        θ_c      = α × 1.4826 × MAD_c
    Each channel calibrates to its own noise floor. Noisy channels
    (poor coupling) get higher thresholds automatically.

    STEP 2 — Bidirectional polarity spikes across multiple timescales
    ────────────────────────────────────────────────────────────────────
    Inspired by: Guo et al. Frontiers Neurosci (2021); sensor SNN
    evaluation (arXiv 2407.09260, 2024).
    For each lag L in lags ([0.5, 1.5, 4.0 ms]):
        ΔX_L[c,t] = X[c,t] − X[c,t−L]
        +vote when ΔX_L >  +θ_c  (compression wave)
        −vote when ΔX_L <  −θ_c  (rarefaction wave)

    STEP 3 — Multi-scale majority vote (per polarity)
    ────────────────────────────────────────────────────
    Candidate spike at (c,t) when pos_votes OR neg_votes ≥ min_votes.
    Real DAS arrivals are coherent across all timescales; noise spikes
    are time-localised (single lag) and fail the vote.

    STEP 4 — Spatial coherence gate (vectorised with scipy.ndimage)
    ─────────────────────────────────────────────────────────────────
    Extends ST-MW spatial condition to AMSTE candidate spikes:
        da_s[c,t] = max(|X[c-ws/2:c+ws/2, t]|) − min(...)
    Keep spike only if da_s > thr_s.
    Suppresses isolated single-channel noise (fading, erratic).

    OUTPUT: binary {0,1} — compatible with all SNN notebooks.
    Sweep best: alpha=3.0, lags=[1,3,8], min_votes=2, ws=16, thr_s=0.5
    → SNR=3.91×, ratio=1.70× (sp_ev/sp_no), prec=0.295
    """
    if lags is None:
        lags = AMSTE_LAGS
    n_tr, n_smp = data.shape

    # Step 1: Per-channel MAD threshold
    med_c   = np.median(data, axis=1, keepdims=True)
    mad_c   = np.median(np.abs(data - med_c), axis=1, keepdims=True)
    theta_c = np.maximum(alpha * 1.4826 * mad_c, 1e-9)   # (n_tr, 1)

    # Steps 2+3: Multi-scale polarity vote
    pos_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    neg_votes = np.zeros((n_tr, n_smp), dtype=np.int32)
    for lag in lags:
        diff_l = data[:, lag:] - data[:, :-lag]
        pos_votes[:, lag:] += (diff_l >  theta_c).astype(np.int32)
        neg_votes[:, lag:] += (diff_l < -theta_c).astype(np.int32)

    candidate = ((pos_votes >= min_votes) |
                 (neg_votes >= min_votes))

    # Step 4: Spatial coherence gate (scipy sliding filter — vectorised)
    abs_data = np.abs(data)
    spat_max = maximum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    spat_min = minimum_filter1d(abs_data, size=ws, axis=0, mode='nearest')
    da_s     = spat_max - spat_min

    out = (candidate & (da_s > thr_s)).astype(np.float32)
    sp  = float(np.sum(out) / out.size)
    print(f"   AMSTE     sparsity: {sp:.2%}  "
          f"(α={alpha}, lags={lags}, mv={min_votes}, ws={ws}ch, θ_s={thr_s})")
    return out

def sfbe_encoding(data, dt_ms,
                   bands=None,
                   ratio_thr=SFBE_RATIO_THR,
                   sta_ms=SFBE_STA_MS):
    """
    Sub-band Frequency Band Encoding (SFBE) — Proposed P7.
    Adapted from: FEEL-SNN NeurIPS 2024 (Xu et al.) frequency encoding;
    STA/LTA onset detection: Allen (1978) BSSA.

    FEEL-SNN original: decomposes image into uniform frequency bands;
    each band assigned to a different time step — time position encodes
    which frequency band fired (mimics selective visual attention).

    DAS modifications:
      1. Seismic-physics sub-bands (task-tuned for FORGE 2024 dataset;
         not universal P/S labels — validate via spectral analysis):
           Band 1: 50–100 Hz   (lower frequency arrivals)
           Band 2: 100–150 Hz  (mixed arrivals)
           Band 3: 150–200 Hz  (higher frequency arrivals)
           Band 4: 200–250 Hz  (highest frequency coda)
      2. STA/LTA firing replaces absolute ATDE threshold (Allen 1978):
           LTA_fixed = mean(env²[0:200ms])  — pre-event baseline window
                       (always before event at 380ms — not contaminated)
           STA[t]    = mean(env²[t−sta_w:t]) — causal short window
           Fire when STA[t]/LTA_fixed > ratio_thr
      3. Time-multiplex retained from FEEL-SNN (band position = wave type)

    EVALUATION NOTE: windowed SNR metric is not applicable to time-
    multiplexed output. Use sparsity discrimination ratio (sp_ev/sp_no)
    as the primary metric. Sweep: ratio_thr=10.0, sta_ms=8.0
    → sp_ev/sp_no=6.41× (highest class separability of all 10 encoders)
    """
    if bands is None:
        bands = SFBE_BANDS
    fs_hz   = 1000.0 / dt_ms
    n_tr, n_smp = data.shape
    n_bands = len(bands)
    seg_len = n_smp // n_bands
    out     = np.zeros((n_tr, n_smp), dtype=np.float32)
    sta_w   = max(1, int(sta_ms / dt_ms))
    pre_end = max(1, int(PRE_EVENT_END_MS / dt_ms))   # 200ms baseline end

    for b_idx, (lo, hi) in enumerate(bands):
        nyq   = 0.5 * fs_hz
        blow  = np.clip(lo / nyq, 1e-6, 0.99)
        bhigh = np.clip(hi / nyq, 1e-6, 0.99)
        b_c, a_c = butter(4, [blow, bhigh], btype='band')

        seg_start = b_idx * seg_len
        seg_end   = seg_start + seg_len if b_idx < n_bands - 1 else n_smp

        for t in range(n_tr):
            filt_b = filtfilt(b_c, a_c, data[t])
            env_sq = np.abs(hilbert(filt_b)) ** 2

            # Fixed LTA: mean energy in pre-event baseline window (0–200ms)
            lta_fixed = max(float(env_sq[:pre_end].mean()), 1e-12)

            # Causal STA using cumsum
            cs  = np.cumsum(env_sq)
            sta = np.zeros(n_smp)
            sta[:sta_w] = cs[:sta_w] / np.arange(1, sta_w + 1)
            sta[sta_w:] = (cs[sta_w:] - cs[:-sta_w]) / sta_w

            # Fire where STA/LTA_fixed > ratio_thr
            ratio = sta / lta_fixed
            for n in range(sta_w, n_smp):
                if ratio[n] > ratio_thr:
                    seg_pos = seg_start + int(n * seg_len / n_smp)
                    if seg_start <= seg_pos < seg_end:
                        out[t, seg_pos] = 1.0

        band_sp = float(np.sum(out[:, seg_start:seg_end]) /
                        max(out[:, seg_start:seg_end].size, 1))
        print(f"   SFBE band {b_idx+1} ({lo:3d}–{hi:3d} Hz)  "
              f"sparsity: {band_sp:.2%}")

    sp = float(np.sum(out) / out.size)
    print(f"   SFBE      total sparsity: {sp:.2%}  "
          f"[evaluation: sp_ev/sp_no ratio, not windowed SNR]")
    return out


# =============================================================================
# BUILD ALL ENCODINGS
# =============================================================================
def _build_all_spike_encodings(norm_data, norm_env, filtered, dt_ms,
                                norm_signed=None):
    """
    Build all 10 spike encodings.

    Input routing — two normalisation schemes:
      norm_env    [0,1]  → Rate, Phase, TTFS (amplitude/envelope encoders)
      norm_signed [-1,1] → Delta-Mod, PDE, ATDE, MASTE, ST-MW, AMSTE
                           (polarity-sensitive encoders)
      filtered   (raw bandpass) → PDE, SFBE (use analytic signal internally)

    If norm_signed is not supplied it falls back to norm_data for
    backwards compatibility, but a warning is printed.
    """
    if norm_signed is None:
        print("      norm_signed not provided — falling back to norm_data.")
        print("      Polarity encoders (AMSTE, ST-MW, MASTE, ATDE, Delta-Mod)")
        print("      require signed [-1,+1] input. Call normalize_signed(filtered).")
        norm_signed = norm_data

    print("\n     Building all 10 encoders …")
    print("   ── Baselines (B1–B3) — use norm_env [0,1] ─────────────────")

    # B1–B3: amplitude/envelope input [0,1]
    rate  = rate_encoding(norm_env,  dt_ms)
    phase = phase_encoding(norm_env, dt_ms)
    ttfs  = ttfs_encoding(norm_env,  dt_ms)

    print("   ── Proposed (P1–P7) ────────────────────────────────────────")
    print("      P1,P3,P4,P5,P6 → norm_signed [-1,+1]")
    print("      P2,P7          → filtered (analytic signal)")

    # P1: Delta-Mod — signed input preserves negative strain events
    dm    = delta_modulation_encoding(norm_signed, dt_ms)

    # P2: PDE — uses filtered directly (Hilbert analytic signal internally)
    pde   = phase_delta_encoding(filtered, dt_ms)

    # P3: ATDE — signed input; EMA threshold tracks both + and − excursions
    atde  = adaptive_threshold_delta_encoding(norm_signed, dt_ms)

    # P4: MASTE — signed input; multi-lag voting on both polarities
    maste = maste_encoding(norm_signed, dt_ms)

    # P5: ST-MW — signed input; spatial gate works on |x| internally
    stmw  = stmw_encoding(norm_signed, dt_ms)

    # P6: AMSTE (DAS-ASE) — signed input REQUIRED for polarity voting
    # step 2 fires pos_vote when ΔX > +θ_c, neg_vote when ΔX < -θ_c
    # with [0,1] input neg_votes ≈ 0 → polarity novelty is lost
    amste = amste_encoding(norm_signed, dt_ms)

    # P7: SFBE — uses filtered directly (sub-band filtering + STA/LTA)
    sfbe  = sfbe_encoding(filtered, dt_ms)

    return {
        'Rate':      rate,
        'Phase':     phase,
        'TTFS':      ttfs,
        'Delta-Mod': dm,
        'PDE':       pde,
        'ATDE':      atde,
        'MASTE':     maste,
        'ST-MW':     stmw,
        'AMSTE':     amste,
        'SFBE':      sfbe,
    }


# =============================================================================
# 4. METRICS
# =============================================================================
def spike_density_profile(spikes, dt_ms, smooth_ms=20.0):
    stacked     = np.mean(np.abs(spikes), axis=0)
    kernel_size = max(3, int(smooth_ms / dt_ms))
    kernel      = np.ones(kernel_size) / kernel_size
    return np.convolve(stacked, kernel, mode='same')

def temporal_fidelity(spikes, envelope, dt_ms, smooth_ms=20.0):
    spike_profile    = spike_density_profile(spikes, dt_ms, smooth_ms)
    envelope_stacked = np.mean(envelope, axis=0)
    kernel_size      = max(3, int(smooth_ms / dt_ms))
    kernel           = np.ones(kernel_size) / kernel_size
    env_smooth       = np.convolve(envelope_stacked, kernel, mode='same')
    r, _             = pearsonr(spike_profile, env_smooth)
    return float(r)

def event_snr(spikes, dt_ms,
              event_start_ms=EVENT_START_MS, event_end_ms=EVENT_END_MS):
    n_tr, n = np.abs(spikes).shape
    s       = max(0, int(event_start_ms / dt_ms))
    e       = min(n, int(event_end_ms   / dt_ms))
    density     = np.mean(np.abs(spikes), axis=0)
    in_event    = np.mean(density[s:e])
    out_samples = np.concatenate([density[:s], density[e:]])
    min_density = 1.0 / (n_tr * n)
    out_event   = max(np.mean(out_samples), min_density) if len(out_samples) > 0 else min_density
    return float(in_event / out_event)

def sparsity_ratio(spikes_ev, spikes_no):
    """sp_ev / sp_no — primary metric for SFBE (time-multiplexed output)."""
    sp_ev = float(np.abs(spikes_ev).mean())
    sp_no = float(np.abs(spikes_no).mean())
    return sp_ev / max(sp_no, 1e-9)

def event_precision(spikes, dt_ms,
                    event_start_ms=EVENT_START_MS, event_end_ms=EVENT_END_MS):
    n = spikes.shape[1]
    s = max(0, int(event_start_ms / dt_ms))
    e = min(n, int(event_end_ms   / dt_ms))
    total = np.sum(np.abs(spikes))
    return float(np.sum(np.abs(spikes[:, s:e])) / total) if total > 0 else 0.0

def onset_detection_latency(spikes, dt_ms,
                             event_start_ms=EVENT_START_MS, search_ms=100.0):
    s = max(0, int(event_start_ms / dt_ms))
    e = min(spikes.shape[1], int((event_start_ms + search_ms) / dt_ms))
    latencies = []
    for t in range(spikes.shape[0]):
        idx = np.where(np.abs(spikes[t, s:e]) > 0)[0]
        if len(idx):
            latencies.append(idx[0] * dt_ms)
    return float(np.mean(latencies)) if latencies else None

def energy_efficiency(spikes, fidelity):
    energy_uj = float(np.sum(np.abs(spikes)) * ENERGY_PER_SPIKE_PJ / 1e6)
    return float(fidelity / (energy_uj + 1e-9))

def compute_all_metrics(spikes, envelope, dt_ms, name):
    fidelity  = temporal_fidelity(spikes, envelope, dt_ms)
    snr       = event_snr(spikes, dt_ms)
    precision = event_precision(spikes, dt_ms)
    latency   = onset_detection_latency(spikes, dt_ms)
    sp        = float(np.sum(np.abs(spikes)) / spikes.size)
    energy_uj = float(np.sum(np.abs(spikes)) * ENERGY_PER_SPIKE_PJ / 1e6)
    eff       = energy_efficiency(spikes, fidelity)
    sfbe_note = '  [use sp_ev/sp_no ratio]' if name == 'SFBE' else ''
    print(f"\n   [{name}] Fidelity={fidelity:.3f}  SNR={snr:.2f}×  "
          f"Precision={precision:.3f}  Sparsity={sp:.2%}  "
          f"Energy={energy_uj:.3f}µJ{sfbe_note}")
    return {'fidelity': fidelity, 'snr': snr, 'precision': precision,
            'latency_ms': latency, 'sparsity': sp, 'energy_uj': energy_uj,
            'efficiency': eff}


# =============================================================================
# 5. FULL ENCODING EXPERIMENT
# =============================================================================
def run_encoding_experiment(all_encodings, norm_envelope, dt_ms):
    print(f"\n{'='*70}")
    print("ENCODING EXPERIMENT — EVENT CHARACTERISATION METRICS")
    print(f"{'='*70}")
    results = {}
    for name, spk in all_encodings.items():
        results[name] = compute_all_metrics(spk, norm_envelope, dt_ms, name)

    print(f"\n{'='*105}")
    print("FINAL COMPARISON — All 10 Encoders (3 Baselines + 7 Proposed)")
    print(f"{'='*105}")
    print(f"{'Encoding':12} | {'Fidelity':>9} | {'SNR':>7} | "
          f"{'Precision':>10} | {'Onset ms':>9} | "
          f"{'Energy µJ':>9} | {'Effic.':>7} | {'Sparsity':>8}")
    print("-" * 105)
    for name, r in results.items():
        lat  = f"{r['latency_ms']:.1f}" if r['latency_ms'] is not None else "N/A"
        note = '*' if name == 'SFBE' else ' '
        print(f"{name:12} | {r['fidelity']:>9.3f} | {r['snr']:>7.2f} | "
              f"{r['precision']:>10.3f} | {lat:>9} | "
              f"{r['energy_uj']:>9.3f} | {r['efficiency']:>7.1f} | "
              f"{r['sparsity']:>7.2%}{note}")
    print(f"\n  * SFBE: SNR metric not applicable to time-multiplexed output.")
    print(f"          Use sp_ev/sp_no = 6.41× as primary SFBE metric.")
    print(f"\n  Sweep best results (400ev+400no files):")
    sweep = {
        'Rate':3.09, 'Phase':2.71, 'TTFS':2.71,
        'Delta-Mod':2.63, 'PDE':2.17, 'ATDE':2.22,
        'MASTE':2.52, 'ST-MW':5.40, 'AMSTE':3.91, 'SFBE':0.83
    }
    for n, snr_v in sweep.items():
        flag = ' ← best SNR' if n == 'ST-MW' else \
               ' ← DAS-ASE (main novelty)' if n == 'AMSTE' else \
               ' ← best sp_ratio (6.41×)' if n == 'SFBE' else ''
        print(f"    {n:12} SNR={snr_v:.2f}×{flag}")
    return results


# =============================================================================
# 6. VISUALISATION
# =============================================================================
def _save(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f"💾 Saved: {name}")
    plt.show()


def plot_segy_section_all_encoders(filtered, norm_data, norm_env, dt_ms,
                                    encodings=None):
    """SEGY section + per-encoder rasters, each saved as its own standalone image.

    Outputs (in OUTPUT_DIR):
        segy_section.png             \u2014 SEGY only, large (7" x 5")
        raster_<encoder>.png         \u2014 one image per encoder (7" x 3")
        segy_all_encoders.png        \u2014 composite reference (kept for legacy)
    """
    if encodings is None:
        encodings = _build_all_spike_encodings(norm_data, norm_env, filtered, dt_ms)

    n_tr, n_smp = filtered.shape
    t_axis      = np.arange(n_smp) * dt_ms
    p1, p99     = np.percentile(filtered, [1, 99])
    vmax        = max(abs(p1), abs(p99))

    # =========================================================================
    # (A) Standalone SEGY section \u2014 prominent strain-rate image
    # =========================================================================
    fig_segy = plt.figure(figsize=(7.0, 5.0))
    ax0 = fig_segy.add_subplot(111)
    im  = ax0.imshow(filtered.T, aspect='auto', cmap='seismic',
                     vmin=-vmax, vmax=vmax,
                     extent=[0, n_tr, t_axis[-1], t_axis[0]])
    ax0.set_title('SEGY section \u2014 bandpass filtered (50\u2013250 Hz)',
                  fontsize=12, fontweight='bold')
    ax0.set_ylabel('Time (ms)', fontsize=11)
    ax0.set_xlabel('Trace index', fontsize=11)
    ax0.tick_params(labelsize=10)
    ax0.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.20,
                label='Event window')
    ax0.legend(fontsize=10, loc='upper right')
    cbar = plt.colorbar(im, ax=ax0, fraction=0.025, pad=0.012)
    cbar.set_label('Amplitude', fontsize=10)
    cbar.ax.tick_params(labelsize=9)
    fig_segy.tight_layout()
    _save(fig_segy, 'segy_section.png')

    # =========================================================================
    # (B) Per-encoder rasters \u2014 each as its own standalone image
    # =========================================================================
    for name, spk in encodings.items():
        fid   = temporal_fidelity(spk, norm_env, dt_ms)
        snr   = event_snr(spk, dt_ms)
        sp    = 100.0 * np.sum(np.abs(spk)) / spk.size
        color = ENCODER_COLORS.get(name, 'gray')

        fig_r = plt.figure(figsize=(7.0, 3.0))
        ax = fig_r.add_subplot(111)
        tr_idx, t_idx = np.where(np.abs(spk) > 0)
        if len(t_idx):
            ax.scatter(tr_idx, t_idx * dt_ms, c=color, s=1.4, alpha=0.60,
                       linewidths=0, rasterized=True)
        else:
            ax.text(0.5, 0.5, 'No spikes', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=11)
        ax.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.20,
                   label='Event window')
        ax.set_xlim(0, n_tr)
        ax.set_ylim(t_axis[-1], t_axis[0])
        ax.set_ylabel('Time (ms)', fontsize=11)
        ax.set_xlabel('Trace index', fontsize=11)
        ax.tick_params(labelsize=10)
        ax.grid(True, alpha=0.25, lw=0.4)
        ax.legend(fontsize=9, loc='upper right')

        sfbe_note = ' [sp_ratio metric]' if name == 'SFBE' else ''
        title = (f"{name}  \u2014  Fidelity={fid:.3f}   SNR={snr:.1f}\u00d7   "
                 f"Sparsity={sp:.2f}%{sfbe_note}")
        ax.set_title(title, fontsize=11.5, color=color, fontweight='bold')
        fig_r.tight_layout()
        safe_name = name.replace(' ', '_').replace('/', '_')
        _save(fig_r, f'raster_{safe_name}.png')

    # =========================================================================
    # (C) Legacy composite \u2014 SEGY + 5x2 grid of rasters
    # =========================================================================
    n_enc  = len(encodings)
    n_rows = (n_enc + 1) // 2
    height_ratios = [4.0] + [1.0] * n_rows
    fig_h = 4.5 + 1.5 * n_rows
    fig = plt.figure(figsize=(7.0, fig_h))
    gs  = gridspec.GridSpec(n_rows + 1, 2,
                            height_ratios=height_ratios,
                            hspace=0.55, wspace=0.20,
                            left=0.09, right=0.97, top=0.95, bottom=0.05)

    ax0 = fig.add_subplot(gs[0, :])
    im  = ax0.imshow(filtered.T, aspect='auto', cmap='seismic',
                     vmin=-vmax, vmax=vmax,
                     extent=[0, n_tr, t_axis[-1], t_axis[0]])
    ax0.set_title('SEGY section \u2014 bandpass filtered (50\u2013250 Hz)',
                  fontsize=10, fontweight='bold')
    ax0.set_ylabel('Time (ms)', fontsize=9)
    ax0.set_xlabel('Trace index', fontsize=9)
    ax0.tick_params(labelsize=8)
    ax0.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.20,
                label='Event window')
    ax0.legend(fontsize=8, loc='upper right')
    cbar = plt.colorbar(im, ax=ax0, fraction=0.020, pad=0.012)
    cbar.set_label('Amplitude', fontsize=8)
    cbar.ax.tick_params(labelsize=7)

    for idx, (name, spk) in enumerate(encodings.items()):
        r = idx // 2 + 1
        c = idx % 2
        fid   = temporal_fidelity(spk, norm_env, dt_ms)
        snr   = event_snr(spk, dt_ms)
        sp    = 100.0 * np.sum(np.abs(spk)) / spk.size
        color = ENCODER_COLORS.get(name, 'gray')

        ax = fig.add_subplot(gs[r, c])
        tr_idx, t_idx = np.where(np.abs(spk) > 0)
        if len(t_idx):
            ax.scatter(tr_idx, t_idx * dt_ms, c=color, s=0.6, alpha=0.55,
                       linewidths=0, rasterized=True)
        else:
            ax.text(0.5, 0.5, 'No spikes', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=8)
        ax.axhspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.18)
        ax.set_xlim(0, n_tr)
        ax.set_ylim(t_axis[-1], t_axis[0])
        ax.tick_params(labelsize=6.5)
        ax.grid(True, alpha=0.22, lw=0.4)
        sfbe_note = ' [sp_ratio]' if name == 'SFBE' else ''
        title = (f"{name}  Fid={fid:.2f}  SNR={snr:.1f}\u00d7  "
                 f"Sp={sp:.2f}%{sfbe_note}")
        ax.set_title(title, fontsize=7.5, color=color, fontweight='bold',
                     pad=2)
        if c == 0:
            ax.set_ylabel('Time (ms)', fontsize=7.5)
        if r == n_rows:
            ax.set_xlabel('Trace index', fontsize=8)

    fig.suptitle('SEGY section + spike rasters \u2014 all 10 encoders '
                 '(3 baselines + 7 proposed | DAS-ASE = AMSTE)',
                 fontsize=10.5, y=0.985, fontweight='bold')
    _save(fig, 'segy_all_encoders.png')
    return fig


def plot_single_trace_all_encoders(filtered, norm_data, norm_env, dt_ms,
                                    trace_idx=DISPLAY_TRACE, encodings=None):
    """Trace waveform + per-encoder spike trains, each saved as its own image.

    Outputs (in OUTPUT_DIR):
        trace_<idx>_waveform.png            \u2014 waveform only (7" x 3.2")
        trace_<idx>_spikes_<encoder>.png    \u2014 one image per encoder (7" x 1.6")
        trace_<idx>_all_encoders.png        \u2014 composite reference (kept)
    """
    if encodings is None:
        encodings = _build_all_spike_encodings(norm_data, norm_env, filtered, dt_ms)

    n_smp   = filtered.shape[1]
    t_axis  = np.arange(n_smp) * dt_ms
    env_tr  = np.abs(hilbert(filtered[trace_idx]))
    env_tr /= (np.max(env_tr) + 1e-9)
    amp_max = np.max(np.abs(filtered[trace_idx]))
    ev_s    = max(0, int(EVENT_START_MS / dt_ms))
    ev_e    = min(n_smp, int(EVENT_END_MS / dt_ms))

    # =========================================================================
    # (A) Standalone waveform
    # =========================================================================
    fig_w = plt.figure(figsize=(7.0, 3.2))
    ax_wave = fig_w.add_subplot(111)
    ax_wave.plot(t_axis, filtered[trace_idx], color='steelblue', lw=0.85,
                 label=f'Trace {trace_idx}')
    ax_wave.fill_between(t_axis, env_tr * amp_max, -env_tr * amp_max,
                         alpha=0.18, color='darkorange', label='Envelope \u00b1')
    ax_wave.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28,
                    label='Event window')
    ax_wave.set_title(f'Trace {trace_idx} \u2014 filtered waveform (50\u2013250 Hz)',
                      fontsize=12, fontweight='bold')
    ax_wave.set_ylabel('Amplitude', fontsize=11)
    ax_wave.set_xlabel('Time (ms)', fontsize=11)
    ax_wave.set_xlim(t_axis[0], t_axis[-1])
    ax_wave.tick_params(labelsize=10)
    ax_wave.legend(fontsize=9.5, loc='upper right', ncol=3)
    ax_wave.grid(True, alpha=0.28)
    fig_w.tight_layout()
    _save(fig_w, f'trace_{trace_idx}_waveform.png')

    # =========================================================================
    # (B) Per-encoder spike trains
    # =========================================================================
    for name, spk in encodings.items():
        color   = ENCODER_COLORS.get(name, 'gray')
        spk_tr  = np.where(np.abs(spk[trace_idx]) > 0)[0]
        times   = spk_tr * dt_ms
        n_total = len(times)
        n_event = int(np.sum(np.abs(spk[trace_idx, ev_s:ev_e]) > 0))
        e_uj    = n_total * ENERGY_PER_SPIKE_PJ / 1e6

        fig_s = plt.figure(figsize=(7.0, 1.6))
        ax = fig_s.add_subplot(111)
        if n_total > 0:
            ax.eventplot(times, lineoffsets=0.5, linelengths=0.85,
                         colors=color, linewidths=1.0)
        else:
            ax.text(0.5, 0.5, '\u2014 0 spikes \u2014', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=10,
                    style='italic')
        ax.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28)
        ax.set_xlim(t_axis[0], t_axis[-1])
        ax.set_yticks([])
        ax.set_ylim(0, 1)
        ax.set_xlabel('Time (ms)', fontsize=10)
        ax.tick_params(labelsize=9)
        ax.grid(True, alpha=0.25, axis='x', lw=0.4)
        title = (f"{name}  \u2014  total={n_total}   in-event={n_event}   "
                 f"E={e_uj:.3f} \u00b5J")
        ax.set_title(title, fontsize=10.5, color=color, fontweight='bold', pad=4)
        fig_s.tight_layout()
        safe_name = name.replace(' ', '_').replace('/', '_')
        _save(fig_s, f'trace_{trace_idx}_spikes_{safe_name}.png')

    # =========================================================================
    # (C) Legacy composite
    # =========================================================================
    n_enc  = len(encodings)
    n_rows = (n_enc + 1) // 2
    height_ratios = [3.2] + [0.8] * n_rows
    fig_h = 4.0 + 1.1 * n_rows
    fig = plt.figure(figsize=(7.0, fig_h))
    gs  = gridspec.GridSpec(n_rows + 1, 2,
                            height_ratios=height_ratios,
                            hspace=0.55, wspace=0.12,
                            left=0.08, right=0.97, top=0.94, bottom=0.06)

    ax_wave = fig.add_subplot(gs[0, :])
    ax_wave.plot(t_axis, filtered[trace_idx], color='steelblue', lw=0.75,
                 label=f'Trace {trace_idx}')
    ax_wave.fill_between(t_axis, env_tr * amp_max, -env_tr * amp_max,
                         alpha=0.18, color='darkorange', label='Envelope \u00b1')
    ax_wave.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28,
                    label='Event window')
    ax_wave.set_title(f'Trace {trace_idx} \u2014 filtered waveform (50\u2013250 Hz)',
                      fontsize=10, fontweight='bold')
    ax_wave.set_ylabel('Amplitude', fontsize=9)
    ax_wave.set_xlabel('Time (ms)', fontsize=9)
    ax_wave.set_xlim(t_axis[0], t_axis[-1])
    ax_wave.tick_params(labelsize=8)
    ax_wave.legend(fontsize=7.5, loc='upper right', ncol=3)
    ax_wave.grid(True, alpha=0.28)

    for idx, (name, spk) in enumerate(encodings.items()):
        r = idx // 2 + 1
        c = idx % 2
        color   = ENCODER_COLORS.get(name, 'gray')
        spk_tr  = np.where(np.abs(spk[trace_idx]) > 0)[0]
        times   = spk_tr * dt_ms
        n_total = len(times)
        n_event = int(np.sum(np.abs(spk[trace_idx, ev_s:ev_e]) > 0))
        e_uj    = n_total * ENERGY_PER_SPIKE_PJ / 1e6

        ax = fig.add_subplot(gs[r, c])
        if n_total > 0:
            ax.eventplot(times, lineoffsets=0.5, linelengths=0.85,
                         colors=color, linewidths=0.8)
        else:
            ax.text(0.5, 0.5, '\u2014 0 spikes \u2014', ha='center', va='center',
                    transform=ax.transAxes, color='gray', fontsize=8,
                    style='italic')
        ax.axvspan(EVENT_START_MS, EVENT_END_MS, color='yellow', alpha=0.28)
        ax.set_xlim(t_axis[0], t_axis[-1])
        ax.set_yticks([])
        ax.set_ylim(0, 1)
        ax.tick_params(labelsize=6.5)
        ax.grid(True, alpha=0.22, axis='x', lw=0.4)
        title = (f"{name}: n={n_total} (ev={n_event})  "
                 f"E={e_uj:.3f} \u00b5J")
        ax.set_title(title, fontsize=7.5, color=color, fontweight='bold',
                     pad=2)
        if r == n_rows:
            ax.set_xlabel('Time (ms)', fontsize=8)

    fig.suptitle(f'Trace {trace_idx} \u2014 spike trains: all 10 encoders '
                 '(3 baselines + 7 proposed | DAS-ASE = AMSTE)',
                 fontsize=10.5, y=0.98, fontweight='bold')
    _save(fig, f'trace_{trace_idx}_all_encoders.png')
    return fig


def plot_temporal_fidelity_comparison(norm_data, norm_envelope, filtered, dt_ms,
                                       encodings=None):
    if encodings is None:
        encodings = _build_all_spike_encodings(norm_data, norm_envelope, filtered, dt_ms)

    t_axis   = np.arange(norm_data.shape[1]) * dt_ms
    env_ref  = np.mean(norm_envelope, axis=0)
    k        = max(3, int(20/dt_ms))
    env_sm   = np.convolve(env_ref, np.ones(k)/k, mode='same')
    colors   = list(ENCODER_COLORS.values())
    n_enc    = len(encodings)

    fig, axs = plt.subplots(n_enc + 1, 1, figsize=(14, 2.5*(n_enc+1)), sharex=True)
    axs[0].plot(t_axis, env_sm, color='black', lw=1.5)
    axs[0].axvspan(EVENT_START_MS, EVENT_END_MS, alpha=0.2, color='yellow',
                   label='Event window')
    axs[0].set_title('Stacked Envelope (Reference)', fontsize=11, fontweight='bold')
    axs[0].set_ylabel('Norm. Amplitude')
    axs[0].legend(fontsize=9)
    axs[0].grid(True, alpha=0.3)

    for i, (name, spk) in enumerate(encodings.items()):
        profile = spike_density_profile(spk, dt_ms)
        fid     = temporal_fidelity(spk, norm_envelope, dt_ms)
        snr     = event_snr(spk, dt_ms)
        sfbe_note = ' *sp_ratio metric*' if name == 'SFBE' else ''
        axs[i+1].fill_between(t_axis, profile, alpha=0.6,
                               color=colors[i % len(colors)])
        axs[i+1].axvspan(EVENT_START_MS, EVENT_END_MS, alpha=0.15, color='yellow')
        axs[i+1].set_title(
            f'{name}  |  Fidelity={fid:.3f}  SNR={snr:.2f}×{sfbe_note}',
            fontsize=11, fontweight='bold')
        axs[i+1].set_ylabel('Spike Density')
        axs[i+1].grid(True, alpha=0.3)

    axs[-1].set_xlabel('Time (ms)', fontsize=11)
    plt.suptitle('Temporal Fidelity: Spike Density vs Event Envelope — '
                 '3 Baselines + 7 Proposed (DAS-ASE=AMSTE)',
                 fontsize=13, y=1.01)
    plt.tight_layout()
    _save(fig, 'temporal_fidelity_comparison_all.png')


def plot_spike_statistics(results):
    names      = list(results.keys())
    fidelity   = [results[n]['fidelity']     for n in names]
    snr        = [results[n]['snr']          for n in names]
    precision  = [results[n]['precision']    for n in names]
    energy     = [results[n]['energy_uj']    for n in names]
    sparsity   = [results[n]['sparsity']*100 for n in names]
    efficiency = [results[n]['efficiency']   for n in names]
    x          = np.arange(len(names))
    w          = 0.55
    colors     = list(ENCODER_COLORS.values())[:len(names)]
    event_fraction = (EVENT_END_MS - EVENT_START_MS) / (WINDOW_END_MS - WINDOW_START_MS)

    fig, axs = plt.subplots(3, 2, figsize=(16, 13))
    for ax, vals, title, hline, hlabel in [
        (axs[0,0], fidelity,   'Temporal Fidelity (↑)',     0.7,  'Target 0.70'),
        (axs[0,1], precision,  'Event Precision (↑)',        event_fraction,
                                                              f'Chance ({event_fraction:.2f})'),
        (axs[1,0], snr,        'Event SNR (↑)',              2.0,  'Target 2×'),
        (axs[1,1], efficiency, 'Efficiency fidelity/µJ (↑)', None, None),
        (axs[2,0], energy,     'Energy Cost µJ (↓)',         None, None),
        (axs[2,1], sparsity,   'Sparsity % (↓)',             None, None),
    ]:
        bars = ax.bar(x, vals, width=w, color=colors)
        ax.set_title(title, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(names, rotation=35, ha='right')
        if hline is not None:
            ax.axhline(hline, ls='--', color='red', label=hlabel)
            ax.legend()
        # Mark ST-MW (best SNR) and AMSTE (main novelty)
        for idx, name in enumerate(names):
            if name == 'ST-MW':
                bars[idx].set_edgecolor('gold')
                bars[idx].set_linewidth(2.5)
            elif name == 'AMSTE':
                bars[idx].set_edgecolor('black')
                bars[idx].set_linewidth(2.5)

    plt.suptitle('Encoding Comparison — 3 Baselines + 7 Proposed\n'
                 'Gold border=ST-MW (best SNR) | Black border=AMSTE (DAS-ASE)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    _save(fig, 'encoding_comparison_metrics_all.png')


# =============================================================================
# 7. MAIN
# =============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("DAS MICROSEISMIC SNN ENCODING — SINGLE FILE PREPROCESSING")
    print("3 Baselines: Rate | Phase | TTFS")
    print("7 Proposed:  Delta-Mod | PDE | ATDE | MASTE | ST-MW | AMSTE | SFBE")
    print("Main novelty: AMSTE (DAS-ASE) — Adaptive Multi-Scale")
    print("              Spatio-Temporal Delta Encoder")
    print("=" * 70)

    if not os.path.exists(FILE_PATH):
        raise FileNotFoundError(f"File not found: {FILE_PATH}")

    data, dt_ms = load_segy_file(FILE_PATH)
    if data is None:
        raise RuntimeError("Failed to load SEGY file")

    print(f"\n Bandpass {FILTER_LOW_HZ}–{FILTER_HIGH_HZ} Hz…")
    filtered = bandpass_filter(data, dt_ms)
    envelope = extract_envelope(filtered)

    if EXTRACT_WINDOW:
        print(f"  Window {WINDOW_START_MS}–{WINDOW_END_MS} ms…")
        filtered, _, _ = extract_time_window(filtered, dt_ms,
                                              WINDOW_START_MS, WINDOW_END_MS)
        envelope, _, _ = extract_time_window(envelope, dt_ms,
                                              WINDOW_START_MS, WINDOW_END_MS)

    norm_data   = normalize_data(filtered, mode='unit')    # [0,1]  legacy
    norm_env    = normalize_data(envelope, mode='unit')    # [0,1]  amplitude encoders
    norm_signed = normalize_signed(filtered)               # [-1,+1] polarity encoders

    stacked    = np.mean(norm_env, axis=0)
    event_time = np.argmax(stacked) * dt_ms
    print(f" Event centre: {event_time:.1f} ms "
          f"(window: {EVENT_START_MS}–{EVENT_END_MS} ms)")
    print(f"   Note: single-file diagnostic window only.")
    print(f"   Batch experiments use per-file labels (not this hard-coded window).")

    print("\n Building all 10 encoders…")
    all_encodings = _build_all_spike_encodings(
        norm_data, norm_env, filtered, dt_ms, norm_signed=norm_signed)

    print("\n Plot 1 — SEGY section + all-encoder rasters…")
    plot_segy_section_all_encoders(filtered, norm_data, norm_env, dt_ms,
                                    encodings=all_encodings)

    print(f"\n Plot 2 — Single-trace spike trains (trace {DISPLAY_TRACE})…")
    plot_single_trace_all_encoders(filtered, norm_data, norm_env, dt_ms,
                                    trace_idx=DISPLAY_TRACE, encodings=all_encodings)

    print("\n Plot 3 — Temporal fidelity comparison…")
    plot_temporal_fidelity_comparison(norm_data, norm_env, filtered, dt_ms,
                                       encodings=all_encodings)

    print("\n Plot 4 — Full encoding experiment…")
    experiment_results = run_encoding_experiment(all_encodings, norm_env, dt_ms)
    print("""
    SINGLE-FILE SNR WARNING:
  SNR values from this one file are NOT representative of batch performance.
  Single-file SNR can be very high (e.g. 20–30×) for unusually clean events.
  Paper-quality metrics from multi-file sweep (N=800 files):
    ST-MW:  SNR=5.40×  (best SNR — spatial-temporal gate)
    AMSTE:  SNR=3.91×  (DAS-ASE main novelty — MAD+polarity+vote+spatial)
    Rate:   SNR=3.09×  (strongest baseline)
  Use batch sweep results for all paper claims, not this single-file result.

  AMSTE POLARITY NOTE (for paper):
  In this diagnostic implementation, positive and negative polarity events
  are fused into a binary spike map {0,1} for compatibility with all
  compared encoders. A polarity-preserving two-channel AMSTE representation
  (output shape 2×C×T) is used in the SNN training phase (notebook 4).
""")

    print("\n Plot 5 — Bar-chart summary…")
    plot_spike_statistics(experiment_results)

    print("\n Saving results…")
    np.save(os.path.join(OUTPUT_DIR, 'all_encodings.npy'), all_encodings)
    np.save(os.path.join(OUTPUT_DIR, 'filtered_data.npy'), filtered)
    np.save(os.path.join(OUTPUT_DIR, 'experiment_results.npy'), experiment_results)
    print(" Done.")